## **Import Libraries**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report,roc_auc_score, RocCurveDisplay
from xgboost import XGBClassifier

: 

## **Load Dataset**

In [ ]:
df = pd.read_csv("hotel_bookings.csv")
df.head()

## **Data Understanding & Exploration**

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
for column in df.columns:
  print(f'The number of unique values for {column} : {df[column].nunique()}')

In [ ]:
categorical_columns = df.select_dtypes(include='object').columns

for column in categorical_columns:
    print(f"\n")
    print(df[column].value_counts())

In [ ]:
missing = df.isnull().sum()

missing[missing > 0]

# **Data Cleaning**

In [ ]:
df[['children', 'country', 'agent', 'company']].head(10)

In [ ]:
df[df['children'].isna()]

### **Handling Missing Values**

In [ ]:
df['children'].median()

In [ ]:
df['children'] = df['children'].fillna(0)

In [ ]:
df['country'] = df['country'].fillna('Unknown')

In [ ]:
df['agent'] = df['agent'].fillna(0)


######The `company` column contains 112,593 missing values, representing approximately 94.3% of the dataset.

In [ ]:
df.drop('company', axis =1, inplace=True)

###**Handling Duplicates**

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace = True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

###**Handling Invalid Values**

In [ ]:
print("Adults:")
print(df['adults'].value_counts().sort_index())

print("\nChildren:")
print(df['children'].value_counts().sort_index())

print("\nBabies:")
print(df['babies'].value_counts().sort_index())

print("\nADR:")
print(df['adr'].describe())

Adults


In [ ]:
df[df['adults'] == 0][
    ['adults', 'children', 'babies', 'adr', 'is_canceled']
]

In [ ]:
df[(df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)].shape

In [ ]:
invalid = (df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)
df = df[~invalid]

Children

In [ ]:
df[df['children'] == 10]

Babies

In [ ]:
df[df['babies'] >= 9]

In [ ]:
df = df[df['babies'] < 9]

Average Daily Rate (adr)

In [ ]:
df[df['adr'] < 0]

In [ ]:
df = df[df['adr'] >= 0]

In [ ]:
df['adr'].min()

In [ ]:
df[df['adr'] > 1000][
    ['hotel', 'adults', 'children', 'babies',
     'stays_in_weekend_nights', 'stays_in_week_nights',
     'market_segment', 'customer_type', 'adr']
]

In [ ]:
df = df[df['adr'] <= 1000]

In [ ]:
df['adr'].max()

##**Final Data Quality Check**

In [ ]:
df.isnull().sum()


In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

In [ ]:
df.describe()

## **Exploratory Data Analysis (EDA)**

In [ ]:
sns.countplot(data=df , x='is_canceled')

plt.title('Booking Cancellation Distribution')
plt.xlabel('Is Canceled')
plt.ylabel('Number of Bookings')
plt.show()

In [ ]:
hotel_counts = df['hotel'].value_counts()

plt.figure(figsize=(7,4))
plt.pie(hotel_counts, labels=hotel_counts.index, autopct='%1.1f%%', startangle=140, explode=(0.05,0.05))
plt.title("Bookings Distribution by Hotel Type", fontsize=16)
plt.show()

In [ ]:
sns.countplot(x='hotel', hue='is_canceled', data=df)

plt.title('Booking Cancellation by Hotel Type')
plt.xlabel('Hotel Type')
plt.ylabel('Number of Bookings')
plt.legend(title='Canceled', labels=['No', 'Yes'])
plt.show()

City Hotel shows a higher cancellation rate than Resort Hotel.

In [ ]:
sns.boxplot(x='is_canceled', y='lead_time', data=df)

plt.title('Lead Time vs Booking Cancellation')
plt.xlabel('Canceled')
plt.ylabel('Lead Time (Days)')
plt.xticks([0, 1], ['Not Canceled', 'Canceled'])
plt.show()

Canceled bookings generally have a higher lead time than non-canceled bookings.

In [ ]:
sns.boxplot(x='is_canceled', y='adr', data=df)

plt.title('ADR vs Booking Cancellation')
plt.xlabel('Canceled')
plt.ylabel('Average Daily Rate')
plt.xticks([0, 1], ['Not Canceled', 'Canceled'])
plt.show()

In [ ]:
sns.countplot(x='market_segment', hue='is_canceled', data=df)

plt.title('Booking Cancellation by Market Segment')
plt.xlabel('Market Segment')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=45)
plt.legend(title='Canceled', labels=['No', 'Yes'])
plt.show()

In [ ]:
sns.countplot(x='deposit_type', hue='is_canceled', data=df)

plt.title('Booking Cancellation by Deposit Type')
plt.xlabel('Deposit Type')
plt.ylabel('Number of Bookings')
plt.legend(title='Canceled', labels=['No', 'Yes'])
plt.show()

In [ ]:
pd.crosstab(df['deposit_type'], df['is_canceled'], normalize='index') * 100

Bookings with a Non Refund deposit type have a significantly higher cancellation rate (94.7%) compared to No Deposit (26.7%) and Refundable bookings (24.3%).

In [ ]:
sns.countplot(x='customer_type', hue='is_canceled', data=df)

plt.title('Booking Cancellation by Customer Type')
plt.xlabel('Customer Type')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=45)
plt.legend(title='Canceled', labels=['No', 'Yes'])
plt.show()

In [ ]:
pd.crosstab(df['customer_type'], df['is_canceled'], normalize='index') * 100

Transient customers have the highest cancellation rate (30.14%), while Group customers have the lowest cancellation rate (9.80%).

In [ ]:
sns.countplot(x='meal', hue='is_canceled', data=df)

plt.title('Booking Cancellation by Meal Type')
plt.xlabel('Meal Type')
plt.ylabel('Number of Bookings')
plt.legend(title='Canceled', labels=['No', 'Yes'])
plt.show()

In [ ]:
pd.crosstab(df['meal'], df['is_canceled'], normalize='index') * 100

SC meal bookings have the highest cancellation rate (35.52%), while Undefined bookings have the lowest rate (16.67%). However, the cancellation rates for BB, HB, and FB are relatively similar.

In [ ]:
sns.barplot(
    data=df,
    x='is_repeated_guest',
    y='is_canceled'
)

plt.title('Cancellation Rate by Repeated Guest Status')
plt.xlabel('Repeated Guest')
plt.ylabel('Cancellation Rate')
plt.show()

In [ ]:
sns.boxplot(x='is_canceled', y='previous_cancellations', data=df)

plt.title('Previous Cancellations by Current Booking Status')
plt.xlabel('Canceled')
plt.ylabel('Previous Cancellations')
plt.show()

In [ ]:
sns.countplot(x='total_of_special_requests', hue='is_canceled', data=df)

plt.title('Cancellation Status by Number of Special Requests')

plt.xlabel('Total Special Requests')
plt.ylabel('Number of Bookings')

plt.show()

In [ ]:
monthly_cancel = df.groupby('arrival_date_month')['is_canceled'].mean()

months_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

monthly_cancel = monthly_cancel.reindex(months_order)

plt.figure(figsize=(10,5))
plt.plot(monthly_cancel.index, monthly_cancel.values, marker='o')

plt.title('Cancellation Rate by Arrival Month')
plt.xlabel('Arrival Month')
plt.ylabel('Cancellation Rate')
plt.xticks(rotation=45)
plt.show()

In [ ]:
yearly_cancel = df.groupby('arrival_date_year')['is_canceled'].mean()

plt.figure(figsize=(8,5))

plt.plot(
    yearly_cancel.index,
    yearly_cancel.values,
    marker='o'
)

plt.title('Cancellation Rate by Arrival Year')
plt.xlabel('Arrival Year')
plt.ylabel('Cancellation Rate')
plt.xticks(yearly_cancel.index)
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    df.select_dtypes(include='number').corr(),
    annot=True,
    fmt='.2f'
)

plt.title('Correlation Heatmap of Numerical Features')
plt.show()

The correlation heatmap shows that most numerical features have weak to moderate correlations. The highest correlation is approximately 0.55, indicating no strong multicollinearity among the numerical features.

## **Data Preprocessing**

In [ ]:
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

In [ ]:
X = X.drop(['reservation_status', 'reservation_status_date'], axis=1)

print(X.shape)

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(exclude='object').columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

In [ ]:
for col in categorical_cols:
    print(f"{col}: {X[col].nunique()} unique values")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

## **Modeling**

### Logistic Regression

In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)

logistic_model.fit(X_train_processed, y_train)

y_pred_logistic = logistic_model.predict(X_test_processed)


print("Accuracy:", accuracy_score(y_test, y_pred_logistic))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

### Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=10
)

dt_model.fit(X_train_processed, y_train)

y_pred_dt = dt_model.predict(X_test_processed)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_rf)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Canceled', 'Canceled']
)

disp.plot()
plt.title('Confusion Matrix - Random Forest')
plt.show()

### XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

xgb_model.fit(X_train_processed, y_train)

y_pred_xgb = xgb_model.predict(X_test_processed)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_xgb = confusion_matrix(y_test, y_pred_xgb)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=['Not Canceled', 'Canceled']
)

disp.plot()
plt.title('Confusion Matrix - Tuned XGBoost')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)
print("ROC-AUC:", roc_auc_xgb)

### **Feature Importance Analysis**

In [ ]:
feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
)

print(feature_importance.head(15))

### **Final Model**

In [ ]:
from sklearn.pipeline import Pipeline

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

final_model.fit(X_train, y_train)

y_pred_final = final_model.predict(X_test)

print("Final Model Accuracy:", accuracy_score(y_test, y_pred_final))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

In [ ]:
import joblib

joblib.dump(final_model, 'hotel_cancellation_model.pkl')

print("Model saved successfully!")

In [ ]:
df.to_csv('hotel_cancellation_data.csv', index=False)

from google.colab import files
files.download('hotel_cancellation_data.csv')